# 🧠 AI-Powered Health Assistant using RAG (Retrieval-Augmented Generation)

**Author:** Muhammad Ali Waris Khan  

### 👨‍💻 Project – AI/ML Engineering  

## 📌 Overview
This project demonstrates a simple Retrieval-Augmented Generation (RAG) system using:
- FAISS for vector search
- HuggingFace embeddings
- Google Gemini for answer generation

The goal is to build a system that answers health-related queries using retrieved knowledge instead of generating responses blindly.

In [1]:
!pip install -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers google-genai


---

## 📂 Step 2: Import Libraries


In [2]:
from google import genai
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

---

## 🔑 Step 3: Setup Gemini API Key

### Get your key from Google AI Studio


In [3]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

client = genai.Client()

---

## 📄 Step 4: Prepare Dataset
#### 📄 Creating a Knowledge Base

We define a small set of health-related documents.  
In real-world systems, this would come from medical articles, PDFs, or databases.


In [4]:
texts = [
    "Heart disease is a leading cause of death worldwide. Regular exercise helps reduce risk.",
    "Diabetes affects how your body processes blood sugar. Healthy diet is important.",
    "High blood pressure can lead to serious complications if untreated.",
    "Mental health includes emotional, psychological, and social well-being.",
    "Regular sleep improves overall health and cognitive function."
]


---

## ✂️ Step 5: Split Text into Chunks

Large texts are split into smaller chunks so that retrieval becomes more accurate and efficient.

In [5]:
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.create_documents(texts)


---

## 🔢 Step 6: Create Embeddings

We convert text into numerical vectors using a pre-trained model.
This allows us to measure similarity between user queries and documents.


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

---

## 🧠 Step 7: Create Vector Store (FAISS)

We store embeddings in FAISS, which allows fast similarity search.

In [7]:
vectorstore = FAISS.from_documents(docs, embeddings)


---

## 🔍 Step 8: Retrieve Relevant Documents

Given a user query, we retrieve the most relevant documents based on similarity.


In [8]:
query = "How can I reduce risk of heart disease?"
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:", doc.page_content)


Document 1: Heart disease is a leading cause of death worldwide. Regular exercise helps reduce risk.
Document 2: High blood pressure can lead to serious complications if untreated.


## 📊 Retrieved Documents Analysis

Below are the top retrieved documents for the query.  
This step is crucial in RAG systems as it ensures that the final answer is grounded in relevant information.

In [9]:
print("\n--- Retrieved Documents with Scores ---\n")

docs_with_scores = vectorstore.similarity_search_with_score(query, k=2)

for i, (doc, score) in enumerate(docs_with_scores):
    print(f"Rank {i+1} | Score: {score:.4f}")
    print(doc.page_content)
    print("-" * 50)


--- Retrieved Documents with Scores ---

Rank 1 | Score: 0.7857
Heart disease is a leading cause of death worldwide. Regular exercise helps reduce risk.
--------------------------------------------------
Rank 2 | Score: 1.3785
High blood pressure can lead to serious complications if untreated.
--------------------------------------------------


---

## 🤖 Step 9: Generate Answer with Gemini

We combine retrieved documents with the user query and pass them to the Gemini model.

This ensures:
- grounded responses
- reduced hallucination

In [10]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [11]:
context = "\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Regular exercise helps reduce risk of heart disease.


## 📈 Final Observations

- The system successfully retrieved relevant documents related to heart disease
- The generated answer was based strictly on retrieved context
- This demonstrates how RAG improves reliability compared to standard LLMs

## ⚠️ Limitations

- Small dataset
- Basic retrieval (no ranking optimization)
- No evaluation metrics

## 🚀 Future Improvements

- Use larger medical datasets
- Add re-ranking models
- Build a user interface (Streamlit)

## ✅ Conclusion

This project demonstrates a simple but effective RAG pipeline combining retrieval and generation.